# GHAP monthly climate–health attribution, forecasts and DHIS2 outputs

Run the cells in order. The notebook follows the GHAP architecture: **data sources → processing (ingestion,
validation, alignment, lag windows) → analytics (exposure–response, lag–response, attribution, forecast and
early warning, model registry, validation and QA) → service tables for DHIS2 dashboards**.

**Publishing is off by default** (`PUBLISH_MODE = 'off'`). With `'off'` the notebook writes the DHIS2 payload
files for review and does not change anything on the server. `'dry_run'` validates on the server, and `'live'`
writes values, but only after a named reviewer is entered in `REVIEWED_BY`.

All modelling code is in the `ghap_engine/` package next to this notebook. Each run is saved without being
overwritten in `ghap_outputs/runs/<run_id>/`. A copy of the latest run is kept in `ghap_outputs/current/`, and `ghap_outputs/model_registry.csv`
plus `ghap_outputs/run_log.jsonl` record every run.

| Change from the previous version | Where |
|---|---|
| Analysis starts in 2015 (trimmed to available data); seasonality by Fourier harmonics and a slow trend instead of 4 df/year | settings, model |
| Minimum-risk reference exposure (MMT) with median/P25/P75 sensitivity; components above/below the reference and extremes | attribution |
| Outcome-specific lag windows (malaria 0–4, cholera 0–2, births third trimester 0–2) plus refits with longer windows | `ghap_engine/config.py` |
| Quasi-Poisson or negative binomial chosen from the data; facility-clustered robust SEs; residual autocorrelation, seasonality and spatial checks | model, QA |
| Disruption terms (COVID-19) and optional confounders; negative-control outcome (live births) for access-to-care effects | settings, QA |
| District heterogeneity (two-stage random effects, BLUPs) and district-mean exposure sensitivity | QA tables |
| Blank components in **submitted** forms count as zero (DHIS2 does not store zeros); unsubmitted forms stay missing | processing |
| Current month and reporting lag excluded; incomplete recent months trimmed | processing |
| Valid aggregation: attributable numbers summed within Monte Carlo draws to district/province/national, month/year/total | aggregates |
| Row accounting (funnel), mapping validation, informative-missingness test, model registry and run log | QA tables |
| **12-month probabilistic forecasts** with alert/epidemic exceedance probabilities and rolling-origin skill | forecasts |
| Review gate, dry run, run id in every DHIS2 value comment, ratios stored with aggregation `NONE` | publishing |

In [7]:
from pathlib import Path  # Work with the project and output folders.
import os  # Read optional environment overrides for the workbook path.
import importlib  # Refresh engine modules cached by an earlier notebook session.
import pandas as pd  # Display the output tables.
from IPython.display import display  # Render tables in the notebook.
import ghap_engine  # Load the GHAP modelling engine package.
from ghap_engine import publish_lifecycle, publish_workflow
from ghap_engine import config, bases, dhis2, inputs, panel, model, forecast, tables, pipeline, publish  # Locate every engine module.
for module in (ghap_engine, config, bases, dhis2, inputs, panel, model, forecast, tables, pipeline, publish_lifecycle, publish_workflow, publish):  # Reload in dependency order.
    importlib.reload(module)  # Pick up any edits made since the kernel started.
from ghap_engine.config import default_settings, analysis_end_month  # Import the settings helpers.
from ghap_engine.dhis2 import make_client  # Import the authenticated DHIS2 client factory.
from ghap_engine.inputs import load_inputs, mapping_validation_table  # Import retrieval and mapping validation.
from ghap_engine.panel import build_panels, completeness_tables, missingness_bias_table  # Import the processing layer.
from ghap_engine.tables import outcome_trends, outcome_seasonality, climate_tables, funnel_table, data_dictionary, TABLE_DESCRIPTIONS  # Import service tables.
from ghap_engine.pipeline import Run, run_attribution, run_forecasts, save_tables, register_run, load_facility_month  # Import orchestration.
from ghap_engine.publish import publish, run_analytics  # Import the gated DHIS2 publisher.
pd.set_option('display.max_columns', 80)  # Show wide tables in full.
pd.set_option('display.width', 250)  # Avoid wrapping wide tables.
print('GHAP engine version', ghap_engine.ENGINE_VERSION)  # Record the engine version used.

GHAP engine version 2.0.0


In [3]:
# import os
# from getpass import getpass
# os.environ["DHIS2_URL"] = input("DHIS2 base URL: ").strip()
# os.environ["DHIS2_USERNAME"] = input("DHIS2 username: ").strip()
# os.environ["DHIS2_PASSWORD"] = getpass("DHIS2 password: ")

## 1. Settings and DHIS2 sign-in

`client = make_client()` reads `DHIS2_URL`, `DHIS2_USER` (or `DHIS2_USERNAME`) and `DHIS2_PASSWORD` from
environment variables first, then falls back to `file.txt` in this folder. The `/me` request checks sign-in
before any data are retrieved. Credentials are never printed.

The analysis ends at the **last complete month minus the reporting lag**. A run in November 2026 therefore
ends in September 2026, because the most recent months are still being reported. Months that are reported
after the end month are not used for fitting. Climate data observed after the end month are used for forecasts.

In [28]:
settings = default_settings()  # Start from the documented engine defaults (ghap_engine/config.py).
settings['workbook'] = os.environ.get('GHAP_WORKBOOK', str(Path('DHIS2 Health Outcomes.xlsx').resolve()))  # Read the mapped numerator/denominator workbook.
settings['start_month'] = os.environ.get('GHAP_START_MONTH', '2015-01')  # Request up to ~12 years; the panel is trimmed to months with data.
settings['output_folder'] = 'ghap_outputs'  # Save runs under ghap_outputs/runs/<run_id> and the latest copy under ghap_outputs/current.
settings['publish_start_month'] = '2024-01'  # Earliest month of attribution values sent to DHIS2.
settings['confounders'] = {}  # Optional {'name': 'uid + uid'} DHIS2 expressions, e.g. RDT stock-out days.
settings['accepted_mapping_warnings'] = []  # outcome_ids whose mapping warning has been reviewed and accepted (see mapping_validation).
settings['n_jobs'] = max(1, min(4, (os.cpu_count() or 2) - 1))  # Model outcomes in parallel processes (identical results; ~15-20 min for 9 x 42 pairs).
REFRESH_DHIS2 = True  # Retrieve fresh inputs; False reuses a saved snapshot with identical settings.
PUBLISH_MODE = 'off'  # 'off' writes payload files only, 'dry_run' validates on the server, 'live' writes values.
REVIEWED_BY = ''  # Name of the reviewer who approved this run; required for PUBLISH_MODE = 'live'.
RUN_ANALYTICS = True  # Start a (server-wide) analytics table update after a live publish.
run = Run(settings)  # Create the versioned run folder, registry identity and run log.
client = make_client()  # Load the DHIS2 credentials into an HTTPS session.
client.get('me.json', {'fields': 'id'})  # Authenticate without displaying credentials.
print('DHIS2 sign-in verified:', client.base)  # Confirm the server URL.
print('Run id:', run.run_id, '| analysis ends:', analysis_end_month(settings))  # Show the run identity and last modelled month.
display(pd.Series({k: v for k, v in settings.items() if k != 'climate_forecast'}, name='setting').astype(str))  # Inspect the configuration.
# Dataset setup automatically discovers every outcome and climate indicator in data.
# Metadata creation does not upload model estimates or change source datasets.
CREATE_OR_UPDATE_DATASETS = True
METADATA_MODE = 'live'  # 'dry_run' validates without creating; 'live' creates/extends both datasets.
settings['publish_facility_params'] = [
    'expected', 'expected_without_exposure',
    'an', 'an_low', 'an_high', 'af', 'af_low', 'af_high',
    'rr', 'rr_low', 'rr_high', 'an_above_ref', 'an_below_ref',
]
settings['publish_complete'] = True
settings['publish_analytics'] = RUN_ANALYTICS


DHIS2 sign-in verified: https://hmis.mohcc.org.zw/nhis
Run id: 20261111_151556_5cd2b029 | analysis ends: 2026-09


workbook                         C:\Users\Ovy\Documents\Scripts\Climate data pu...
workbook_sheet                                                                 NaN
workbook_columns                                                                {}
input_dataset                                                          qqND8igwWqz
start_month                                                                2015-01
end_month                                                                      NaN
reporting_lag_months                                                             1
tail_reporting_ratio                                                           0.8
district_level                                                                   3
province_level                                                                   2
zero_fill_when_form_submitted                                                 True
minimum_denominator                                                              1
max_

## 2. Data sources, ingestion and validation

The study units are the point-geolocated facilities assigned to climate dataset `qqND8igwWqz`. Health values
come from the analytics API. Reporting records (`ACTUAL_REPORTS`) for each source dataset are used to tell a
true zero from a missing value: DHIS2 does not store zeros, so a blank component in a **submitted** form is 0,
and a component in an unsubmitted form is missing. Mapping problems are flagged, not ignored.

In [29]:
inputs_snapshot = load_inputs(client, settings, refresh=REFRESH_DHIS2)  # Retrieve or reload the complete input snapshot.
mapping_validation = mapping_validation_table(inputs_snapshot['outcomes'])  # Parsed operands, names, lag windows and warnings per indicator.
display(mapping_validation)  # Review every mapping before modelling.
print('Facilities:', len(inputs_snapshot['facilities']), '| climate variables:', len(inputs_snapshot['climate_vars']))  # Show the study size.

Metadata: 48 climate variables; 10 workbook outcomes; 60 point facilities
Retrieved health batch 10 / 12
Retrieved health batch 12 / 12
Retrieved climate batch 10 / 21
Retrieved climate batch 20 / 21
Retrieved climate batch 21 / 21
Retrieved reporting batch 6 / 6
Retrieved 17168 health values, 133962 climate values, 39824 reporting records


,outcome_id,indicator_name,source_numerator,numerator_names,source_denominator,denominator_names,denominator_type,lag_max,alt_lag_max,lag_rationale,is_negative_control,mapping_warning
0,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),oFoqROvVKG4 + mH8dkeEipQr,T5 - Malaria - Confirmed Cases (Tested Positiv...,eXRBmcuBdhL + VR2BkDjKjTo,VHW - 01 ( 2021 v1 )Malaria(Number of patients...,mapped_population,4,6,vector breeding and incubation delay (0-4 months),False,
1,preterm_births_aa1ca4,Preterm births,L0LCW2QUbQ6 + BLWTkEZiVGG + vExekjkGFz9,HIS - D3 - Preterm births - 22 to 27 weeks ; ...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbqO...,HIS - D3 - Live Births at Hospital/Clinic - be...,mapped_population,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,
2,his_d3_total_still_birth_4dc56c,HIS - D3 - Total Still births at health facility,vF0stCsDlQd + MZOUUFZa4QS,HIS - D3 - Still births at Health facility (fr...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbqO...,HIS - D3 - Live Births at Hospital/Clinic - be...,mapped_population,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,denominator label mentions ['still'] but mappe...
3,mnch_proportion_of_newbo_0a5cf0,MNCH-Proportion of newborn babies with lowbirt...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbq...,HIS - D3 - Live Births at Hospital/Clinic - be...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbqO...,HIS - D3 - Live Births at Hospital/Clinic - be...,mapped_population,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,
4,his_d3_grand_total_mater_9e67f0,HIS - D3 - Grand Total Maternal Deaths,nKnT9drXf2Z + MUssDRKQpPt,HIS - D3 - Maternal Deaths - At home; HIS - D3...,1,none (count model),none_count_model,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,
5,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,ovdoEVomNeY + e8XNSshWqws + cbqOrDl2Eh2 + War...,HIS - D3 - Live Births at Hospital/Clinic - be...,1,none (count model),none_count_model,2,8,third trimester (lags 0-2); full pregnancy (0-...,True,
6,his_d3_number_of_women_w_7a0cc1,HIS - D3 - Number of women with obstetric Comp...,GdEeADNbv9R,HIS - D3 - Number of women with obstetric Comp...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbqO...,HIS - D3 - Live Births at Hospital/Clinic - be...,mapped_population,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,
7,his_d3_number_of_women_w_c4e6d6,HIS - D3 - Number of women with obstetric Comp...,s9LIAKyEf3x,HIS - D3 - Number of women with obstetric Comp...,ovdoEVomNeY + isn4O2irk4h + e8XNSshWqws + cbqO...,HIS - D3 - Live Births at Hospital/Clinic - be...,mapped_population,2,8,third trimester (lags 0-2); full pregnancy (0-...,False,
8,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,rm3deGvRr8l,RDNS - Cholera Cases,1,none (count model),none_count_model,2,4,waterborne transmission responds within weeks,False,
9,hiv2024v6_clients_tested_7b75cf,HIV2024v6 - Clients tested HIV positive (HF & ...,VHqfP78iXEF+B942bay4lYa,HIV2024v6 - Clients tested HIV positive; HIV20...,1,none (count model),none_count_model,2,4,default short window,False,


Facilities: 60 | climate variables: 48


In [30]:
data = build_panels(inputs_snapshot, settings)  # Build facility-month panels with exclusion reasons and climate cubes.
completeness_facility_month, completeness_summary = completeness_tables(data)  # Reporting status per facility-month and per facility.
missingness_bias = missingness_bias_table(data)  # Compare climate in reported and unreported months.
display(completeness_facility_month.groupby(['indicator_name', 'status']).size().unstack(fill_value=0))  # Exclusion reasons by outcome.
display(completeness_summary.groupby('indicator_name')['pct_valid'].describe().round(1))  # Distribution of facility completeness.
display(missingness_bias.loc[missingness_bias['informative_missingness_flag']].head(20))  # Pairs where missing months had different climate.

status,denominator_below_minimum,denominator_not_reported,numerator_not_reported,ratio_above_max,valid
indicator_name,,,,,
HIS - D3 - Grand Total Maternal Deaths,0,0,6886,0,1574
HIS - D3 - Number of women with obstetric Complications - Pre-eclampsia/eclampsia,196,0,6885,0,1379
HIS - D3 - Number of women with obstetric Complications - Sepsis,196,0,6885,0,1379
HIS - D3 - Total Live births at Hospital/clinic Male&Female,0,0,6885,0,1575
HIS - D3 - Total Still births at health facility,197,0,6885,0,1378
HIV2024v6 - Clients tested HIV positive (HF & VHW),0,0,7226,0,1234
MNCH-Proportion of newborn babies with lowbirth weight <1000g,197,1,6884,0,1378
Malaria positive cases (VHW & HFs),176,1,6448,0,1835
Preterm births,197,0,6885,3,1375


,count,mean,std,min,25%,50%,75%,max
indicator_name,,,,,,,,
HIS - D3 - Grand Total Maternal Deaths,60.0,18.6,2.5,0.0,19.1,19.1,19.1,19.1
HIS - D3 - Number of women with obstetric Complications - Pre-eclampsia/eclampsia,60.0,16.3,4.7,0.0,16.1,18.4,19.1,19.1
HIS - D3 - Number of women with obstetric Complications - Sepsis,60.0,16.3,4.7,0.0,16.1,18.4,19.1,19.1
HIS - D3 - Total Live births at Hospital/clinic Male&Female,60.0,18.6,2.5,0.0,19.1,19.1,19.1,19.1
HIS - D3 - Total Still births at health facility,60.0,16.3,4.7,0.0,16.1,18.4,19.1,19.1
HIV2024v6 - Clients tested HIV positive (HF & VHW),60.0,14.6,1.9,0.0,14.9,14.9,14.9,14.9
MNCH-Proportion of newborn babies with lowbirth weight <1000g,60.0,16.3,4.7,0.0,16.1,18.4,19.1,19.1
Malaria positive cases (VHW & HFs),60.0,21.7,9.6,0.0,14.9,23.4,31.2,31.9
Preterm births,60.0,16.3,4.7,0.0,16.1,18.4,19.1,19.1


,outcome_id,climate_uid,n_reported,n_missing,mean_exposure_reported,mean_exposure_missing,std_mean_difference,p_value,informative_missingness_flag
3,malaria_positive_cases_v_4fb09c,EhIFhX6hLtB,223,19,28.539910,29.310526,0.204306,4.376931e-01,True
5,malaria_positive_cases_v_4fb09c,gbKsMzTJfEC,1835,273,31.505777,32.196703,0.216717,6.385827e-04,True
6,malaria_positive_cases_v_4fb09c,Uwzxq29k1Hj,1835,272,31.640954,32.570956,0.234835,4.358667e-04,True
8,malaria_positive_cases_v_4fb09c,EWFN7735S4o,223,19,38.893274,43.826316,0.876751,1.218335e-03,True
9,malaria_positive_cases_v_4fb09c,IOf0l6awINE,1831,272,94.978291,96.193934,0.213149,1.429014e-03,True
13,malaria_positive_cases_v_4fb09c,U8xj2V5914T,1832,272,4.930240,2.179779,-0.389526,1.644517e-09,True
14,malaria_positive_cases_v_4fb09c,oQJShAjpHuo,222,19,21.053604,19.584211,-0.355508,1.195953e-01,True
15,malaria_positive_cases_v_4fb09c,EnXwcCBStp7,1831,272,25.300765,19.946324,-0.436649,1.261723e-12,True
18,malaria_positive_cases_v_4fb09c,flDdgBClNPI,1835,273,12.793924,11.426740,-0.317325,1.295857e-06,True
19,malaria_positive_cases_v_4fb09c,obs8kCgrtDZ,1834,273,0.259324,0.210256,-0.245660,7.241274e-06,True


## 3. Descriptive views: trends, seasonality and climate hazards

District and higher totals use valid facility-months scaled to full reporting. Months with less than 50%
of facilities reporting are left blank. The hazard thresholds are the pooled P10 and P90 of each climate variable.

In [31]:
outcome_trend_table = outcome_trends(data)  # Monthly observed counts, denominators and rates by level.
outcome_seasonality_table = outcome_seasonality(outcome_trend_table)  # Monthly climatology of each outcome.
climate_views = climate_tables(data, settings)  # Climate trends, climatology, hazard thresholds and hazard flags.
display(outcome_seasonality_table.query("level == 'national'").pivot_table(index='calendar_month', columns='indicator_name', values='mean_cases').round(1))  # National seasonal profile.
display(climate_views['hazard_thresholds'])  # P10/P90 thresholds used for hazard components.

indicator_name,HIS - D3 - Grand Total Maternal Deaths,HIS - D3 - Number of women with obstetric Complications - Pre-eclampsia/eclampsia,HIS - D3 - Number of women with obstetric Complications - Sepsis,HIS - D3 - Total Live births at Hospital/clinic Male&Female,HIS - D3 - Total Still births at health facility,HIV2024v6 - Clients tested HIV positive (HF & VHW),MNCH-Proportion of newborn babies with lowbirth weight <1000g,Malaria positive cases (VHW & HFs),Preterm births,RDNS - Cholera Cases
calendar_month,,,,,,,,,,
1,5.1,20.2,1.8,1822.9,47.3,191.2,233.4,1557.7,153.0,22.9
2,1.5,46.1,4.0,1675.9,40.8,162.7,187.0,1346.4,155.3,33.1
3,2.5,51.7,1.6,1822.4,37.2,188.6,225.9,3410.5,139.8,34.6
4,2.0,24.2,1.2,1880.4,42.1,175.4,213.8,5152.5,249.5,15.0
5,2.6,34.2,2.4,1950.0,45.2,170.8,226.5,5885.9,196.8,5.1
6,5.8,64.3,4.3,1952.0,53.6,174.9,222.4,2824.2,166.5,0.6
7,3.4,27.3,12.4,1923.2,45.3,181.4,192.9,997.0,174.5,0.2
8,2.4,28.7,3.3,1944.6,62.8,149.2,242.4,401.2,218.1,0.2
9,5.8,39.2,11.2,1942.4,51.2,190.7,220.4,626.7,171.6,0.2


,climate_uid,climate_name,low_hazard_threshold,high_hazard_threshold
0,C7z4qm2rPUC,Tg_max,28.00,38.50
1,CPAO2XMrleH,Relative humidity (ERA5-Land),41.40,79.80
2,CRk21XQPsvI,Dewpoint temperature (ERA5-Land),5.09,18.10
3,EBJyjf9KSjM,THI_mean,59.10,72.60
4,EWFN7735S4o,Max heat stress (ERA5-Land),32.40,46.62
5,EhIFhX6hLtB,Heat stress (ERA5-Land),23.77,32.80
6,EnXwcCBStp7,Min humidity (ERA5-Land),10.20,43.90
7,EtOB6DxiV2q,WBGT_in_max,18.60,26.30
8,HaliHVSSi1y,Heat Stress (ERA5-Heat),17.20,25.30
9,IGIrB0U2CB6,Max solar radiation (ERA5-Land),802.75,1120.70


## 4. Exposure–response, lag–response and attribution

Each outcome × climate pair is fitted as a DLNM:

`log E[Y_it] = log N_it + α_facility + Fourier(month) + ns(trend) + events + confounders + cb(x_{i,t-0..L})`

* **Family**: Poisson, then quasi-Poisson or negative binomial, chosen by the Cameron–Trivedi variance check.
* **Standard errors**: clustered by facility (when there are 20 or more facilities), so they are robust to within-facility autocorrelation.
* **Reference**: the minimum-risk exposure (MMT) on the pooled cumulative curve, searched between P5 and P95.
* **Attribution**: backward, `AF_t = 1 − exp(−(cb_t − cb_ref)·β)` and `AN_t = AF_t × observed_t`. The 95% CIs come from 1,000 Monte Carlo draws.
* **Aggregation**: district, province and national totals are sums **within each draw**, so their intervals are valid. Never average ratios or interval bounds.
* **QA**: blocking flags are `fit_failed`, `rank_deficient`, `non_finite_estimates`, `implausible_effect` and `uninformative_ci`. Other flags are warnings that reviewers should read.

In [32]:
results = run_attribution(data, settings, run, missingness_bias)  # Fit every pair, attribute, run diagnostics, heterogeneity and sensitivity.
T = results['tables']  # Collect the attribution tables.
model_summary = T['model_summary']  # One row per outcome x climate pair.
display(model_summary['status'].value_counts().rename('pairs').to_frame())  # Fitted and unfitted pair counts.
display(model_summary.groupby('indicator_name')['qa_pass'].agg(['sum', 'size']).rename(columns={'sum': 'qa_pass', 'size': 'pairs'}))  # QA-passed pairs per outcome.
display(model_summary.query('qa_pass').sort_values('wald_p')[['indicator_name', 'climate_name', 'family', 'mmt_percentile', 'rr_p10', 'rr_p10_low', 'rr_p10_high', 'rr_p90', 'rr_p90_low', 'rr_p90_high', 'paf', 'paf_low', 'paf_high', 'wald_p', 'qa_flags']].head(20))  # Strongest associations.

Modelled outcome 1 / 10 (36s)
Modelled outcome 2 / 10 (36s)
Modelled outcome 3 / 10 (36s)
Modelled outcome 4 / 10 (36s)
Modelled outcome 5 / 10 (46s)
Modelled outcome 6 / 10 (53s)
Modelled outcome 7 / 10 (53s)
Modelled outcome 8 / 10 (53s)
Modelled outcome 9 / 10 (67s)
Modelled outcome 10 / 10 (67s)


,pairs
status,
fitted,314
insufficient_data,138
degenerate_exposure,26
fit_failed,2


,qa_pass,pairs
indicator_name,,
HIS - D3 - Grand Total Maternal Deaths,3,48
HIS - D3 - Number of women with obstetric Complications - Pre-eclampsia/eclampsia,10,48
HIS - D3 - Number of women with obstetric Complications - Sepsis,1,48
HIS - D3 - Total Live births at Hospital/clinic Male&Female,35,48
HIS - D3 - Total Still births at health facility,0,48
HIV2024v6 - Clients tested HIV positive (HF & VHW),0,48
MNCH-Proportion of newborn babies with lowbirth weight <1000g,35,48
Malaria positive cases (VHW & HFs),26,48
Preterm births,22,48


,indicator_name,climate_name,family,mmt_percentile,rr_p10,rr_p10_low,rr_p10_high,rr_p90,rr_p90_low,rr_p90_high,paf,paf_low,paf_high,wald_p,qa_flags
260,HIS - D3 - Total Live births at Hospital/clini...,Precipitation (ERA5-Land),negative_binomial,5.0,1.000000,1.000000,1.000000,1.040785,0.961742,1.126324,0.011568,-0.012220,0.034151,7.171061e-216,short_series;residual_spatial_clustering;infor...
6,Malaria positive cases (VHW & HFs),Max air temperature (ERA5-Land),negative_binomial,95.0,8.516087,5.768082,12.573286,1.710409,1.544622,1.893991,0.776350,0.722772,0.817209,6.760508e-108,residual_seasonality;access_effect_in_negative...
26,Malaria positive cases (VHW & HFs),Tdb_max,negative_binomial,95.0,8.713398,5.838599,13.003685,1.717573,1.548628,1.904948,0.779985,0.722375,0.826222,1.598251e-106,residual_seasonality;access_effect_in_negative...
29,Malaria positive cases (VHW & HFs),Tg_max,negative_binomial,95.0,9.446410,6.243822,14.291672,1.647572,1.503886,1.804986,0.778533,0.726913,0.818497,9.779700e-105,residual_seasonality;access_effect_in_negative...
15,Malaria positive cases (VHW & HFs),Min humidity (ERA5-Land),negative_binomial,5.0,1.083398,0.976442,1.202070,5.465278,3.971719,7.520487,0.699290,0.576072,0.794683,6.171330e-101,residual_autocorrelation;residual_seasonality;...
30,Malaria positive cases (VHW & HFs),Tg_mean,negative_binomial,95.0,8.546541,4.826153,15.134905,1.556987,1.421945,1.704854,0.603164,0.528978,0.657436,3.151584e-84,residual_seasonality;access_effect_in_negative...
0,Malaria positive cases (VHW & HFs),Air temperature (ERA5-Land),negative_binomial,95.0,8.363852,4.681115,14.943879,1.508539,1.387282,1.640394,0.590671,0.516439,0.651935,1.117958e-83,residual_seasonality;access_effect_in_negative...
27,Malaria positive cases (VHW & HFs),Tdb_mean,negative_binomial,95.0,8.363852,4.681115,14.943879,1.508539,1.387282,1.640394,0.590671,0.516568,0.643515,1.117958e-83,residual_seasonality;access_effect_in_negative...
21,Malaria positive cases (VHW & HFs),Relative humidity (ERA5-Land),negative_binomial,5.0,1.014500,0.906164,1.135789,4.730101,2.937995,7.615348,0.679374,0.546398,0.769656,2.696583e-76,residual_seasonality
13,Malaria positive cases (VHW & HFs),Min dewpoint temperature (ERA5-Land),negative_binomial,5.0,1.452916,1.003480,2.103643,7.365879,4.378101,12.392630,0.799929,0.640961,0.882677,5.305338e-71,residual_seasonality;residual_spatial_clusteri...


In [33]:
aggregates = T['aggregates']  # Attributable numbers and fractions by level and period.
display(aggregates.query("level == 'national' and period_type == 'year'").merge(model_summary.query('qa_pass')[['outcome_id', 'climate_uid']]).head(20))  # National annual burden (QA-passed pairs).
display(T['exposure_response'].query("reference_type == 'mmt'").head(23))  # Exposure-response curve rows for the first pair.
display(T['lag_response'].head(12))  # Lag-response rows for the first pair.
display(T['reference_sensitivity'].head(8))  # PAF under alternative reference exposures.
display(T['heterogeneity'].head(8) if len(T['heterogeneity']) else 'Fewer than two districts with enough data for heterogeneity.')  # District effects and BLUPs.
display(T['sensitivity'].head(9) if len(T['sensitivity']) else 'Sensitivity refits disabled.')  # Alternative lag window and district-mean exposure.
display(T['negative_control'])  # Utilisation tracer associations (access-to-care check).
facility_month_example = load_facility_month(results, next(iter(data['outcomes'])))  # Facility-month attribution for the first outcome.
display(facility_month_example.head(10))  # Inspect facility-month RR, AF, AN and components.

,level,indicator_name,org_unit_id,climate_name,org_unit_name,period_type,period,facility_months,observed,expected,denominator,an,an_low,an_high,an_above_ref,an_above_ref_low,an_above_ref_high,an_below_ref,an_below_ref_low,an_below_ref_high,an_extreme_high,an_extreme_high_low,an_extreme_high_high,an_extreme_low,an_extreme_low_low,an_extreme_low_high,paf,paf_low,paf_high,attributable_rate_per_1000,attributable_rate_per_1000_low,attributable_rate_per_1000_high,outcome_id,climate_uid
0,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Air temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2023,348.0,21052.0,22488.465756,89175.0,13756.559654,12256.761060,14997.042163,-184.092314,-283.828006,-96.977514,13870.894358,12413.423950,15111.523511,-184.092314,-283.828006,-96.977514,-1.852157e+02,-417.553445,-12.361886,0.653456,0.582214,0.712381,154.264756,137.446158,168.175415,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
1,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Air temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2024,456.0,7801.0,7393.752235,53436.0,3602.941924,3086.889557,4028.659229,-376.621790,-550.739015,-238.052061,3915.724911,3411.212573,4362.681580,-376.621790,-550.739015,-238.052061,-4.056929e+00,-8.213062,-1.391488,0.461856,0.395704,0.516429,67.425367,57.767974,75.392230,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
2,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Air temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2025,573.0,17686.0,15903.356754,83967.0,10240.272336,8586.371143,11523.514111,-428.742868,-744.592162,-162.740311,10513.633026,8993.767456,11779.004541,-428.742868,-744.592162,-162.740311,4.644627e+01,-185.539220,221.110522,0.579004,0.485490,0.651561,121.955915,102.258880,137.238603,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
3,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Air temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2026,458.0,16818.0,17245.093097,75484.0,9823.373684,8683.272070,10783.193823,-71.152603,-128.110847,-19.808002,9877.962548,8730.579956,10839.579395,-71.152603,-128.110847,-19.808002,-4.700586e+02,-746.746129,-260.169836,0.584099,0.516308,0.641170,130.138489,115.034604,142.854034,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
4,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Dewpoint temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2023,348.0,21052.0,20644.308712,89175.0,17103.657943,14564.550854,18558.941260,17103.558406,14564.112964,18558.898389,0.103951,-0.334096,0.508276,9430.271082,7317.466931,10756.501758,1.039506e-01,-0.334096,0.508276,0.812448,0.691837,0.881576,191.798799,163.325494,208.118206,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
5,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Dewpoint temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2024,454.0,7800.0,8465.506359,53407.0,5690.641711,4579.426196,6375.158484,5684.602036,4557.903052,6377.298486,5.646298,-35.777029,44.891856,1051.337094,707.762590,1292.944467,5.646298e+00,-35.777029,44.891856,0.729569,0.587106,0.817328,106.552357,85.745804,119.369345,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
6,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Dewpoint temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2025,573.0,17686.0,17735.297997,83967.0,13610.881585,11215.160815,14998.446362,13609.579745,11206.825903,15001.468018,1.203206,-5.547036,6.992911,8307.492239,6033.743213,9799.485571,1.203206e+00,-5.547036,6.992911,0.769585,0.634126,0.848041,162.097986,133.566288,178.623108,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
7,national,Malaria positive cases (VHW & HFs),uLFWBf4qXuv,Dewpoint temperature (ERA5-Land),zw Zimbabwe Ministry of Health,year,2026,458.0,16818.0,16510.566967,75484.0,14554.857839,12617.337524,15477.673193,14554.857839,12617.337524,15477.673193,0.000000,0.000000,0.000000,8187.839829,6553.444067,9120.360864,0.000000e+00,0.000000,0.000000,0.865433,0.750228,0.920304,192.820437,167.152482,205.045742,

,reference_type,indicator_name,reference_value,climate_name,exposure_percentile,exposure_value,rr,rr_low,rr_high,outcome_id,climate_uid
0,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),1.0,13.900,5.606512,2.439355,12.885777,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
1,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),2.5,14.400,6.064245,2.775679,13.249038,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
2,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),5.0,15.100,6.718536,3.276841,13.775073,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
3,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),10.0,17.200,8.363852,4.681115,14.943879,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
4,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),15.0,18.200,8.667133,5.082617,14.779628,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
5,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),20.0,18.900,8.581527,5.197552,14.168708,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
6,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),25.0,19.600,8.219926,5.162631,13.087741,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
7,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),30.0,20.100,7.794277,5.046146,12.039041,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
8,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),35.0,20.600,7.242295,4.856787,10.799492,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
9,mmt,Malaria positive cases (VHW & HFs),27.7,Air temperature (ERA5-Land),40.0,21.100,6.588057,4.600207,9.434901,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ


,exposure_percentile,indicator_name,exposure_value,climate_name,lag,rr,rr_low,rr_high,outcome_id,climate_uid,reference_value
0,1,Malaria positive cases (VHW & HFs),13.9,Air temperature (ERA5-Land),0,0.193114,0.133038,0.280319,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
1,1,Malaria positive cases (VHW & HFs),13.9,Air temperature (ERA5-Land),1,1.610303,1.213163,2.137450,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
2,1,Malaria positive cases (VHW & HFs),13.9,Air temperature (ERA5-Land),2,4.353810,3.087686,6.139116,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
3,1,Malaria positive cases (VHW & HFs),13.9,Air temperature (ERA5-Land),3,3.816809,2.887928,5.044457,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
4,1,Malaria positive cases (VHW & HFs),13.9,Air temperature (ERA5-Land),4,1.084927,0.675231,1.743206,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
5,10,Malaria positive cases (VHW & HFs),17.2,Air temperature (ERA5-Land),0,0.429506,0.332710,0.554464,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
6,10,Malaria positive cases (VHW & HFs),17.2,Air temperature (ERA5-Land),1,1.618273,1.354927,1.932803,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
7,10,Malaria positive cases (VHW & HFs),17.2,Air temperature (ERA5-Land),2,3.053568,2.517000,3.704521,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
8,10,Malaria positive cases (VHW & HFs),17.2,Air temperature (ERA5-Land),3,2.885606,2.405779,3.461135,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7
9,10,Malaria positive cases (VHW & HFs),17.2,Air temperature (ERA5-Land),4,1.365652,1.049043,1.777817,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,27.7


,reference_type,indicator_name,reference_value,climate_name,an,an_low,an_high,paf,paf_low,paf_high,outcome_id,climate_uid
0,mmt,Malaria positive cases (VHW & HFs),27.700,Air temperature (ERA5-Land),37423.147599,32720.035107,41304.624219,0.590671,0.516439,0.651935,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
1,median,Malaria positive cases (VHW & HFs),21.800,Air temperature (ERA5-Land),-81770.429564,-111090.782812,-59636.615430,-1.290630,-1.753410,-0.941279,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
2,p25,Malaria positive cases (VHW & HFs),19.600,Air temperature (ERA5-Land),-149817.339983,-240766.069141,-84849.885547,-2.364653,-3.800149,-1.339235,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
3,p75,Malaria positive cases (VHW & HFs),24.200,Air temperature (ERA5-Land),-9407.402058,-17404.080908,-3573.450098,-0.148482,-0.274699,-0.056402,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
4,mmt,Malaria positive cases (VHW & HFs),4.388,Dewpoint temperature (ERA5-Land),50960.039078,43468.768066,55403.970605,0.804344,0.686103,0.874487,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
5,median,Malaria positive cases (VHW & HFs),12.300,Dewpoint temperature (ERA5-Land),31931.533579,20401.853711,37941.984863,0.504002,0.322019,0.598870,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
6,p25,Malaria positive cases (VHW & HFs),8.100,Dewpoint temperature (ERA5-Land),48456.707203,36549.727734,53861.920996,0.764832,0.576894,0.850147,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
7,p75,Malaria positive cases (VHW & HFs),16.700,Dewpoint temperature (ERA5-Land),-21897.050566,-71174.929492,2890.993945,-0.345619,-1.123413,0.045631,malaria_positive_cases_v_4fb09c,CRk21XQPsvI


,district_id,indicator_name,district_name,climate_name,facilities,events,log_rr_p10,se_p10,log_rr_p90,se_p90,rr_p10,rr_p10_low,rr_p10_high,blup_rr_p10,blup_rr_p10_low,blup_rr_p10_high,pooled_rr_p10,i2_p10,q_p10,q_pvalue_p10,tau2_p10,rr_p90,rr_p90_low,rr_p90_high,blup_rr_p90,blup_rr_p90_low,blup_rr_p90_high,pooled_rr_p90,i2_p90,q_p90,q_pvalue_p90,tau2_p90,outcome_id,climate_uid
0,M9m516j0wdc,Malaria positive cases (VHW & HFs),me Uzumba Maramba Pfungwe District,Air temperature (ERA5-Land),1,561.0,11.790737,2.345163,-0.400382,0.522455,1.320237e+05,1331.708468,1.308864e+07,2476.842130,74.266352,82604.662697,49.886258,0.812016,26.598011,0.000068,5.401573,6.700643e-01,0.240654,1.865690e+00,1.521438,1.312556,1.763561,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
1,MGUTPgm9GAU,Malaria positive cases (VHW & HFs),mw Makonde District,Air temperature (ERA5-Land),2,210.0,1.911632,2.992223,0.369466,1.481589,6.764116e+00,0.019195,2.383651e+03,23.520762,0.445048,1243.072370,49.886258,0.812016,26.598011,0.000068,5.401573,1.446962e+00,0.079305,2.640045e+01,1.537366,1.324773,1.784076,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
2,O5dih53Gqoc,Malaria positive cases (VHW & HFs),mv Masvingo District,Air temperature (ERA5-Land),5,115.0,14.857000,7.512736,5.773582,4.037701,2.833434e+06,1.141151,7.035304e+12,129.790114,0.880700,19127.363180,49.886258,0.812016,26.598011,0.000068,5.401573,3.216878e+02,0.117618,8.798228e+05,1.539279,1.326231,1.786551,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
3,WSgzOybiqyJ,Malaria positive cases (VHW & HFs),mc Mount Darwin District,Air temperature (ERA5-Land),22,47215.0,2.154567,0.156523,0.413976,0.029087,8.624159e+00,6.345726,1.172066e+01,8.692776,6.399119,11.808556,49.886258,0.812016,26.598011,0.000068,5.401573,1.512821e+00,1.428986,1.601574e+00,1.517598,1.437377,1.602296,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
4,Yd5BmwJToLp,Malaria positive cases (VHW & HFs),mi Chirumhanzu District,Air temperature (ERA5-Land),22,78.0,27.061490,14.556668,11.303206,6.381861,5.657908e+11,0.230022,1.391687e+24,88.699038,0.487683,16132.440734,49.886258,0.812016,26.598011,0.000068,5.401573,8.108120e+04,0.299620,2.194167e+10,1.538952,1.325931,1.786195,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
5,mk2qcpB2N56,Malaria positive cases (VHW & HFs),mc Mbire District,Air temperature (ERA5-Land),6,15178.0,0.665137,0.751405,0.460332,0.037350,1.944757e+00,0.445920,8.481523e+00,2.643728,0.636390,10.982730,49.886258,0.812016,26.598011,0.000068,5.401573,1.584600e+00,1.472742,1.704954e+00,1.571033,1.468631,1.680576,1.537515,0.386127,8.145014,1.484227e-01,0.003499,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ
6,M9m516j0wdc,Malaria positive cases (VHW & HFs),me Uzumba Maramba Pfungwe District,Dewpoint temperature (ERA5-Land),1,561.0,1.730923,0.624663,16.689198,2.518936,5.645863e+00,1.659610,1.920678e+01,2.148555,1.105646,4.175194,1.591835,0.853362,34.097651,0.000002,0.121128,1.770218e+07,127017.850297,2.467110e+09,1990.860678,116.826958,33926.469533,86.239423,0.891739,46.184895,8.327976e-09,2.190541,malaria_positive_cases_v_4fb09c,CRk21XQPsvI
7,MGUTPgm9GAU,Malaria positive cases (VHW & HFs),mw Makonde District,Dewpoint temperature (ERA5-Land),2,210.0,1.599367,0.379508,6.377430,3.121760,4.949900e+00,2.352624,1.041455e+01,2.672840,1.550611,4.607263,1.591835,0.853362,34.097651,0.000002,0.121128,5.884134e+02,1.295350,2.672870e+05,122.676524,6.082864,2474.086326,86.239423,0.891739,46.184895,8.327976e-09,2.190541,malaria_positive_cases_v_4fb09c,CRk21XQPsvI


,specification,indicator_name,lag_max,climate_name,exposure_scale,rr_p10,rr_p10_low,rr_p10_high,rr_p90,rr_p90_low,rr_p90_high,paf,paf_low,paf_high,mmt,an,rows,mmt_percentile,an_low,an_high,outcome_id,climate_uid,error
0,main,Malaria positive cases (VHW & HFs),4,Air temperature (ERA5-Land),facility,8.363852,4.681115,14.943879,1.508539,1.387282,1.640394,0.590671,0.516439,0.651935,27.700000,37423.147599,1835.0,NaN,NaN,NaN,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,NaN
1,alternative_lag_window,Malaria positive cases (VHW & HFs),6,Air temperature (ERA5-Land),facility,10.809440,3.699128,31.586898,1.444562,1.330473,1.568435,0.529205,0.339879,0.656028,27.700000,33528.815534,1835.0,95.0,21533.685254,41563.938477,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,NaN
2,district_mean_exposure,Malaria positive cases (VHW & HFs),4,Air temperature (ERA5-Land),district_mean,5.549612,2.892225,10.648617,1.339472,1.260530,1.423359,0.557963,0.478890,0.618324,26.910455,35350.840334,1835.0,95.0,30341.049268,39175.161621,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,NaN
3,main,Malaria positive cases (VHW & HFs),4,Dewpoint temperature (ERA5-Land),facility,1.018393,0.782577,1.325268,10.311301,4.550519,23.365015,0.804344,0.686103,0.874487,4.388000,50960.039078,1833.0,NaN,NaN,NaN,malaria_positive_cases_v_4fb09c,CRk21XQPsvI,NaN
4,alternative_lag_window,Malaria positive cases (VHW & HFs),6,Dewpoint temperature (ERA5-Land),facility,1.297458,0.706023,2.384339,34.426655,12.053689,98.326296,0.889969,0.755164,0.949363,4.200000,56383.996820,1831.0,5.0,47843.431934,60146.887207,malaria_positive_cases_v_4fb09c,CRk21XQPsvI,NaN
5,district_mean_exposure,Malaria positive cases (VHW & HFs),4,Dewpoint temperature (ERA5-Land),district_mean,1.033309,0.922625,1.157271,26.895808,12.845335,56.314955,0.874164,0.819172,0.904834,6.517391,55384.382514,1835.0,14.0,51900.300293,57327.560840,malaria_positive_cases_v_4fb09c,CRk21XQPsvI,NaN
6,main,Malaria positive cases (VHW & HFs),4,Heatwave threshold (ERA5-Land),facility,5.769520,4.085955,8.146777,1.237492,1.166594,1.312700,0.510891,0.422085,0.578639,37.150000,32368.533433,1835.0,NaN,NaN,NaN,malaria_positive_cases_v_4fb09c,gbKsMzTJfEC,NaN
7,alternative_lag_window,Malaria positive cases (VHW & HFs),6,Heatwave threshold (ERA5-Land),facility,5.448324,3.690045,8.044409,1.267128,1.155163,1.389945,0.590239,0.527178,0.643301,37.150000,37395.798084,1835.0,95.0,33400.444727,40757.590918,malaria_positive_cases_v_4fb09c,gbKsMzTJfEC,NaN
8,district_mean_exposure,Malaria positive cases (VHW & HFs),4,Heatwave threshold (ERA5-Land),district_mean,4.823734,3.411926,6.819731,1.129227,1.062928,1.199662,0.440829,0.312046,0.537758,36.333333,27929.618690,1835.0,95.0,19770.279443,34070.748633,malaria_positive_cases_v_4fb09c,gbKsMzTJfEC,NaN


,outcome_id,indicator_name,climate_uid,climate_name,rr_p10,rr_p10_low,rr_p10_high,rr_p90,rr_p90_low,rr_p90_high,wald_p,access_effect_detected
240,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,nyzCb9pjlkQ,Air temperature (ERA5-Land),1.051885,1.002346,1.103872,1.325328,0.982931,1.786996,9.988055e-02,True
241,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,CRk21XQPsvI,Dewpoint temperature (ERA5-Land),1.118780,0.760717,1.645380,1.016952,0.979372,1.055974,7.834132e-01,False
245,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,gbKsMzTJfEC,Heatwave threshold (ERA5-Land),1.005504,0.975197,1.036753,1.215498,0.978319,1.510177,9.761141e-02,False
246,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,Uwzxq29k1Hj,Max air temperature (ERA5-Land),1.053544,0.993779,1.116903,1.303954,1.056464,1.609423,1.844520e-01,True
247,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,OFnaoduqdkj,Max dewpoint temperature (ERA5-Land),1.213969,0.873749,1.686664,1.020794,0.982986,1.060057,4.703667e-01,False
249,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,IOf0l6awINE,Max humidity (ERA5-Land),1.094429,0.953103,1.256710,1.018518,0.907412,1.143228,7.364919e-02,False
250,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,rtwTPRTsSoc,Max precipitation (ERA5-Land),1.155637,0.998869,1.337009,1.032490,0.911257,1.169853,3.690303e-01,False
251,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,IGIrB0U2CB6,Max solar radiation (ERA5-Land),1.679943,1.091398,2.585866,1.104223,0.997886,1.221892,1.305178e-03,True
252,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,Uq8KjwlVSZF,Max wind speed (ERA5-Land),1.013655,0.981763,1.046584,1.065700,0.951577,1.193509,1.306480e-01,False
253,his_d3_total_live_births_f17cc1,HIS - D3 - Total Live births at Hospital/clini...,U8xj2V5914T,Min dewpoint temperature (ERA5-Land),1.116622,0.879272,1.418042,1.018864,0.985174,1.053707,4.616408e-01,False


,facility_id,facility_name,district_id,district_name,province_id,province_name,period,year,observed,denominator,expected,exposure_lag0,exposure_window_mean,reference_exposure,rr,rr_low,rr_high,af,af_low,af_high,an,an_low,an_high,an_above_ref,an_above_ref_low,an_above_ref_high,an_below_ref,an_below_ref_low,an_below_ref_high,an_extreme_high,an_extreme_high_low,an_extreme_high_high,an_extreme_low,an_extreme_low_low,an_extreme_low_high,expected_without_exposure,outcome_id,climate_uid,climate_name
0,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202304,2023,3.0,13.0,2.676149,20.3,21.97,27.7,3.276045,2.407436,4.458051,0.694754,0.584620,0.775687,2.084262,1.753861,2.327060,-2.065015e-14,-3.130829e-14,-9.992007e-15,2.084262,1.753861,2.327060,-2.065015e-14,-3.130829e-14,-9.992007e-15,-2.065015e-14,-3.130829e-14,-9.992007e-15,0.816884,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
1,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202305,2023,9.0,21.0,5.153256,19.4,21.24,27.7,3.759257,2.618040,5.397938,0.733990,0.618035,0.814744,6.605910,5.562313,7.332697,-6.195044e-14,-9.392487e-14,-2.997602e-14,6.605910,5.562313,7.332697,-6.195044e-14,-9.392487e-14,-2.997602e-14,-6.195044e-14,-9.392487e-14,-2.997602e-14,1.370818,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
2,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202306,2023,4.0,18.0,3.368851,18.4,20.38,27.7,4.460925,2.956150,6.731678,0.775831,0.661722,0.851449,3.103325,2.646889,3.405795,-2.753353e-14,-4.174439e-14,-1.332268e-14,3.103325,2.646889,3.405795,-2.753353e-14,-4.174439e-14,-1.332268e-14,-2.753353e-14,-4.174439e-14,-1.332268e-14,0.755191,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
3,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202309,2023,1.0,8.0,0.969650,22.2,18.76,27.7,22.527392,13.105180,38.723877,0.955610,0.923694,0.974176,0.955610,0.923694,0.974176,-6.883383e-15,-1.043610e-14,-3.330669e-15,0.955610,0.923694,0.974176,-6.883383e-15,-1.043610e-14,-3.330669e-15,7.308886e-01,6.489832e-01,7.936824e-01,0.043043,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
4,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202310,2023,10.0,24.0,3.645641,22.6,19.40,27.7,19.404897,11.875095,31.709222,0.948467,0.915790,0.968463,9.484666,9.157902,9.684634,-6.883383e-14,-1.043610e-13,-3.330669e-14,9.484666,9.157902,9.684634,-6.883383e-14,-1.043610e-13,-3.330669e-14,7.042346e+00,6.280572e+00,7.648101e+00,0.187872,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
5,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202402,2024,1.0,7.0,1.064583,23.6,23.39,27.7,3.414250,2.835552,4.111051,0.707110,0.647335,0.756753,0.707110,0.647335,0.756753,-6.883383e-15,-1.043610e-14,-3.330669e-15,0.707110,0.647335,0.756753,-6.883383e-15,-1.043610e-14,-3.330669e-15,-6.883383e-15,-1.043610e-14,-3.330669e-15,0.311806,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
6,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202407,2024,0.0,3.0,0.263495,17.9,19.44,27.7,5.039636,3.165558,8.023209,0.801573,0.684100,0.875362,0.000000,0.000000,0.000000,-0.000000e+00,-0.000000e+00,-0.000000e+00,0.000000,0.000000,0.000000,-0.000000e+00,-0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.052285,malaria_positive_cases_v_4fb09c,nyzCb9pjlkQ,Air temperature (ERA5-Land)
7,AD1DBy9TnBQ,mv Shonganiso Clinic,O5dih53Gqoc,mv Masvingo District,nVljIjXbw6R,mv Masvingo Province,202408,2024,0.0,3.0,0.273241,19.3,18.60,27.7,12.321393,7.189469,21.116542,0.918840,0.860908,0.952644,0.000000,0.000000,0.000000,-0.000000e+00,-0.000000e+00,-0.000000e+00,0.

## 5. Forecasts for the next 12 months and early warning

For each outcome, the engine forecasts the 12 months after the last analysed month.

* **Climate inputs**: observed climate is used where it already exists. Unknown months are filled from
  *analog years* (the same months in each past year), which keeps the seasonal cycle and the spatial
  pattern. Seasonal forecast members, such as ECMWF SEAS5, can be supplied in `settings['climate_forecast']`.
* **Uncertainty**: predictive draws combine coefficient uncertainty, climate uncertainty and outcome noise (NB or quasi-Poisson).
* **Thresholds**: alert = calendar-month P75 of history; epidemic = calendar-month mean + 2 SD. Each forecast row reports the probability of exceeding both, and a risk level.
* **Model selection**: rolling-origin evaluation (4 origins, 1–12 month leads) scores every climate model
  against a no-climate model and a seasonal-naive forecast. `forecast_outcome` uses the climate model with
  the best out-of-sample CRPS **only if it beats the no-climate model**; otherwise it uses the no-climate model.
* **Full set**: `forecast_pair` keeps the forecasts from every QA-passed climate model.

In [34]:
forecasts = run_forecasts(data, settings, results, run)  # Forecast 12 months ahead, evaluate skill and select models.
forecast_outcome = forecasts['forecast_outcome']  # Selected-model forecasts at every level.
display(forecast_outcome.query("level == 'national'")[['indicator_name', 'period', 'lead_months', 'forecast_mean', 'forecast_pi80_low', 'forecast_pi80_high', 'forecast_pi95_low', 'forecast_pi95_high', 'climate_attributable_cases', 'prob_exceed_alert', 'prob_exceed_epidemic', 'risk_level', 'selected_model', 'selected_climate_name']])  # National 12-month outlook.
display(forecast_outcome.query("level == 'district' and risk_level in ['high', 'moderate']")[['indicator_name', 'org_unit_name', 'period', 'forecast_mean', 'alert_threshold', 'prob_exceed_alert', 'prob_exceed_epidemic', 'risk_level']].head(30))  # District alerts.
skill = forecasts['forecast_skill']  # Out-of-sample forecast skill.
display(skill.query("lead_band == 'all' and forecast_type in ['no_climate_baseline', 'seasonal_naive', 'operational']").sort_values(['indicator_name', 'crps']).groupby('indicator_name').head(4) if len(skill) else 'No rolling-origin evaluation was possible.')  # Best models per outcome.

,indicator_name,period,lead_months,forecast_mean,forecast_pi80_low,forecast_pi80_high,forecast_pi95_low,forecast_pi95_high,climate_attributable_cases,prob_exceed_alert,prob_exceed_epidemic,risk_level,selected_model,selected_climate_name
852,Malaria positive cases (VHW & HFs),202610,1,790.857,495.0,1160.1,392.775,1418.175,659.582011,0.000,0.0,low,climate_model,Heatwave threshold (ERA5-Land)
853,Malaria positive cases (VHW & HFs),202611,2,399.800,242.0,589.1,187.950,752.150,256.607070,0.000,0.0,low,climate_model,Heatwave threshold (ERA5-Land)
854,Malaria positive cases (VHW & HFs),202612,3,361.861,221.9,525.1,171.975,670.150,142.319964,0.000,0.0,low,climate_model,Heatwave threshold (ERA5-Land)
855,Malaria positive cases (VHW & HFs),202701,4,694.224,412.0,1022.0,328.000,1265.075,167.684659,0.000,0.0,low,climate_model,Heatwave threshold (ERA5-Land)
856,Malaria positive cases (VHW & HFs),202702,5,1103.744,635.8,1648.3,505.950,2154.275,78.021077,0.016,0.0,low,climate_model,Heatwave threshold (ERA5-Land)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7771,RDNS - Cholera Cases,202705,8,0.000,0.0,0.0,0.000,0.000,0.000000,0.000,0.0,low,no_climate_baseline,
7772,RDNS - Cholera Cases,202706,9,0.000,0.0,0.0,0.000,0.000,0.000000,0.000,0.0,low,no_climate_baseline,
7773,RDNS - Cholera Cases,202707,10,0.000,0.0,0.0,0.000,0.000,0.000000,0.000,0.0,low,no_climate_baseline,
7774,RDNS - Cholera Cases,202708,11,0.000,0.0,0.0,0.000,0.000,0.000000,0.000,0.0,low,no_climate_baseline,


,indicator_name,org_unit_name,period,forecast_mean,alert_threshold,prob_exceed_alert,prob_exceed_epidemic,risk_level
721,Malaria positive cases (VHW & HFs),mi Chirumhanzu District,202610,5.008,4.714286,0.501,0.129,moderate
722,Malaria positive cases (VHW & HFs),mw Makonde District,202610,15.748,4.500000,0.878,0.753,high
723,Malaria positive cases (VHW & HFs),mc Mount Darwin District,202610,589.381,456.222222,0.709,0.209,moderate
724,Malaria positive cases (VHW & HFs),mc Mbire District,202610,166.408,86.500000,0.839,0.709,high
726,Malaria positive cases (VHW & HFs),mv Masvingo District,202611,4.868,3.125000,0.618,0.003,moderate
728,Malaria positive cases (VHW & HFs),mw Makonde District,202611,6.422,3.000000,0.645,0.382,moderate
729,Malaria positive cases (VHW & HFs),mc Mount Darwin District,202611,275.381,168.812500,0.863,0.434,moderate
732,Malaria positive cases (VHW & HFs),mv Masvingo District,202612,4.134,3.125000,0.516,0.000,moderate
738,Malaria positive cases (VHW & HFs),mv Masvingo District,202701,5.797,3.125000,0.691,0.003,moderate
740,Malaria positive cases (VHW & HFs),mw Makonde District,202701,6.772,4.500000,0.581,0.343,moderate


,outcome_id,indicator_name,model,origins,scored_rows,crps,mae,bias,coverage_80,coverage_95,skill_vs_baseline,lead_band,climate_uid,forecast_type,climate_name
34,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),gbKsMzTJfEC|operational,4,2343,15.913542,20.807742,-7.876914,0.831413,0.927443,0.116215,all,gbKsMzTJfEC,operational,Heatwave threshold (ERA5-Land)
24,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),Uwzxq29k1Hj|operational,4,2343,16.265150,20.755294,-8.949861,0.754161,0.862569,0.096688,all,Uwzxq29k1Hj,operational,Max air temperature (ERA5-Land)
0,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),C7z4qm2rPUC|operational,4,2343,16.328450,20.740282,-9.339270,0.746479,0.858728,0.093172,all,C7z4qm2rPUC,operational,Tg_max
32,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),gKG6agUih5g|operational,4,2343,16.343405,21.180420,-8.593271,0.755015,0.8758,0.092342,all,gKG6agUih5g,operational,THI_max
55,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,seasonal_naive,4,2802,0.440043,0.440043,0.423626,NaN,NaN,0.036548,all,,seasonal_naive,
54,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,no_climate_baseline,4,2832,0.456736,1.592768,1.589237,0.98976,0.99541,0.000000,all,,no_climate_baseline,


## 6. Row accounting, data dictionary, registry and saved tables

Every table below can be used as a DHIS2 data-visualisation source or imported as DHIS2 data. The data
dictionary describes each column. The funnel shows where facility-months are lost, from the full grid
through to the QA-passed pairs.

In [35]:
funnel = funnel_table(data, results['counts'], model_summary)  # Row accounting by outcome and climate variable.
display(funnel.groupby('indicator_name').agg(grid_facility_months=('grid_facility_months', 'first'), valid_outcome_rows=('valid_outcome_rows', 'first'), max_rows_with_exposure=('rows_with_exposure_history', 'max'), qa_pass_pairs=('qa_pass', 'sum')))  # Funnel summary.
output_tables = {  # Every table saved for review and DHIS2 dashboards.
    'mapping_validation': mapping_validation, 'completeness_facility_month': completeness_facility_month, 'completeness_summary': completeness_summary,  # Data quality.
    'missingness_bias': missingness_bias, 'outcome_trends': outcome_trend_table, 'outcome_seasonality': outcome_seasonality_table,  # Data quality and descriptive views.
    **climate_views, **T, **forecasts, 'funnel': funnel,  # Climate views, attribution tables, forecast tables and row accounting.
}  # End of the table collection.
output_tables['run_log'] = run.log_table()  # Events logged so far.
dictionary_sources = dict(output_tables, attribution_facility_month=facility_month_example.head(1))  # Include the per-outcome facility-month files in the dictionary.
output_tables['data_dictionary'] = data_dictionary(dictionary_sources, TABLE_DESCRIPTIONS)  # Column descriptions for every table.
table_index = save_tables(run, output_tables)  # Write parquet/CSV into the run folder and refresh ghap_outputs/current.
registry = register_run(run, inputs_snapshot, model_summary, data['end_month'])  # Append this run to the model registry.
display(table_index)  # Saved tables and sizes.
display(registry.tail(5))  # Latest registry entries.

,grid_facility_months,valid_outcome_rows,max_rows_with_exposure,qa_pass_pairs
indicator_name,,,,
HIS - D3 - Grand Total Maternal Deaths,8460,1574,1574,3
HIS - D3 - Number of women with obstetric Complications - Pre-eclampsia/eclampsia,8460,1379,1379,10
HIS - D3 - Number of women with obstetric Complications - Sepsis,8460,1379,1379,1
HIS - D3 - Total Live births at Hospital/clinic Male&Female,8460,1575,1575,35
HIS - D3 - Total Still births at health facility,8460,1378,1378,0
HIV2024v6 - Clients tested HIV positive (HF & VHW),8460,1234,1234,0
MNCH-Proportion of newborn babies with lowbirth weight <1000g,8460,1378,1378,35
Malaria positive cases (VHW & HFs),8460,1835,1835,26
Preterm births,8460,1375,1375,22


,table,rows,columns,csv
0,mapping_validation,10,12,True
1,completeness_facility_month,84600,8,True
2,completeness_summary,600,8,True
3,missingness_bias,480,9,True
4,outcome_trends,24044,15,True
5,outcome_seasonality,8279,11,True
6,climate_trends,158261,17,True
7,climate_seasonality,39138,10,True
8,hazard_thresholds,48,4,True
9,hazard_events,131339,12,True


,run_id,started,finished,engine_version,settings_hash,input_snapshot,data_hash,data_retrieved,analysis_end_month,pairs,pairs_fitted,pairs_qa_pass,status,published,reviewed_by
0,20261110_102411_f00e5e7b,2026-11-10T10:24:11,2026-11-10T11:00:04,2.0.0,f00e5e7b,ecfbf4de3a11,4b2353b6ff6b,2026-11-10T10:40:48,202609,432,71,35,completed,False,
1,20261110_141320_4f4fca2a,2026-11-10T14:13:20,2026-11-10T14:17:22,2.0.0,4f4fca2a,ecfbf4de3a11,33a4a3d6fc11,2026-11-10T14:15:17,202609,432,315,137,completed,False,
2,20261110_142903_5cd2b029,2026-11-10T14:29:03,2026-11-10T14:32:07,2.0.0,5cd2b029,ecfbf4de3a11,33a4a3d6fc11,2026-11-10T14:30:07,202609,432,315,137,completed,False,
3,20261111_084405_5cd2b029,2026-11-11T08:44:05,2026-11-11T08:45:43,2.0.0,5cd2b029,996dbdcf7622,46f67aacf2da,2026-11-11T08:44:40,202609,480,71,35,completed,True,Obvious J Kawanzaruwa
4,20261111_151556_5cd2b029,2026-11-11T15:15:56,2026-11-11T15:19:07,2.0.0,5cd2b029,996dbdcf7622,6fc84772e80a,2026-11-11T15:16:39,202609,480,314,132,completed,False,


In [36]:
from ghap_engine.pipeline import load_facility_month

model_summary = results["tables"]["model_summary"]
display(model_summary)

# Choose an outcome with a successfully fitted model.
outcome_id = model_summary.loc[
    model_summary["status"].eq("fitted"), "outcome_id"
].iloc[0]

historical_results = load_facility_month(results, outcome_id)

display(historical_results[[
    "facility_name", "period", "climate_name",
    "observed", "expected", "expected_without_exposure",
    "rr", "af", "an", "an_low", "an_high"
]].head(20))

,outcome_id,indicator_name,climate_uid,climate_name,denominator_type,is_negative_control,lag_max,rows_analysis,facilities,months,first_period,last_period,events,status,qa_flags,qa_pass,error,rows_used_in_fit,facilities_without_events,family,dispersion,nb_alpha,covariance,trend_df,exposure_df,event_terms,parameters,qaic,pseudo_r2,mmt,mmt_percentile,extreme_low_cut,extreme_high_cut,rr_p1,rr_p1_low,rr_p1_high,rr_p10,rr_p10_low,rr_p10_high,rr_p90,rr_p90_low,rr_p90_high,rr_p99,rr_p99_low,rr_p99_high,wald_chi2,wald_df,wald_p,total_an,total_an_low,total_an_high,paf,paf_low,paf_high,residual_acf_lag1,residual_acf_lag12,residual_seasonality_p,morans_i,morans_i_p,heterogeneity_i2_p90,heterogeneity_q_pvalue_p90
0,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),nyzCb9pjlkQ,Air temperature (ERA5-Land),mapped_population,False,4,1835,58,45,202301,202609,63357.0,fitted,residual_seasonality;access_effect_in_negative...,True,,1683.0,8.0,negative_binomial,2.813183,0.035013,cluster_robust_facility,1.0,3.0,,64.0,11807.671247,0.649416,27.700,95.0,17.2,26.30,5.606512,2.439355,12.885777,8.363852,4.681115,14.943879,1.508539,1.387282,1.640394,0.500713,0.405864,0.617728,414.469510,9.0,1.117958e-83,37423.147599,32720.022021,41304.628027,0.590671,0.516439,0.651935,0.275744,-0.036962,5.454235e-09,0.046822,0.074,0.386127,1.484227e-01
1,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),CRk21XQPsvI,Dewpoint temperature (ERA5-Land),mapped_population,False,4,1833,58,45,202301,202609,63356.0,fitted,residual_autocorrelation;residual_seasonality;...,True,,1681.0,8.0,negative_binomial,7.094838,0.000001,cluster_robust_facility,1.0,3.0,,64.0,17660.017309,0.611825,4.388,5.5,5.6,19.08,1.123482,0.388337,3.250294,1.018393,0.782577,1.325268,10.311301,4.550519,23.365015,14.403156,5.987456,34.647585,316.972326,9.0,6.520718e-63,50960.039078,43468.772266,55403.985449,0.804344,0.686103,0.874487,0.333173,-0.065123,6.859981e-09,0.119727,0.002,0.891739,8.327976e-09
2,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),HaliHVSSi1y,Heat Stress (ERA5-Heat),mapped_population,False,4,44,22,2,202605,202606,4948.0,insufficient_data,,False,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),EhIFhX6hLtB,Heat stress (ERA5-Land),mapped_population,False,4,109,7,20,202405,202512,2747.0,insufficient_data,,False,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),wvIjCYpPeng,Heatwave count (ERA5-Land),mapped_population,False,4,1809,58,45,202301,202609,62275.0,degenerate_exposure,,False,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,hiv2024v6_clients_tested_7b75cf,HIV2024v6 - Clients tested HIV positive (HF & ...,sXvWIgZHwLv,WBGT_in_min,none_count_model,False,2,1229,59,21,202501,202609,3613.0,insufficient_data,,False,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
476,hiv2024v6_clients_tested_7b75cf,HIV2024v6 - Clients tested HIV positive (HF & ...,oBkSd3RutAJ,WBGT_out_max,none_count_model,False,2,1234,59,21,202501,202609,3619.0,insufficient_data,,False,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

,facility_name,period,climate_name,observed,expected,expected_without_exposure,rr,af,an,an_low,an_high
0,mv Shonganiso Clinic,202304,Air temperature (ERA5-Land),3.0,2.676149,0.816884,3.276045,0.694754,2.084262,1.753861,2.327060
1,mv Shonganiso Clinic,202305,Air temperature (ERA5-Land),9.0,5.153256,1.370818,3.759257,0.733990,6.605910,5.562313,7.332697
2,mv Shonganiso Clinic,202306,Air temperature (ERA5-Land),4.0,3.368851,0.755191,4.460925,0.775831,3.103325,2.646889,3.405795
3,mv Shonganiso Clinic,202309,Air temperature (ERA5-Land),1.0,0.969650,0.043043,22.527392,0.955610,0.955610,0.923694,0.974176
4,mv Shonganiso Clinic,202310,Air temperature (ERA5-Land),10.0,3.645641,0.187872,19.404897,0.948467,9.484666,9.157902,9.684634
5,mv Shonganiso Clinic,202402,Air temperature (ERA5-Land),1.0,1.064583,0.311806,3.414250,0.707110,0.707110,0.647335,0.756753
6,mv Shonganiso Clinic,202407,Air temperature (ERA5-Land),0.0,0.263495,0.052285,5.039636,0.801573,0.000000,0.000000,0.000000
7,mv Shonganiso Clinic,202408,Air temperature (ERA5-Land),0.0,0.273241,0.022176,12.321393,0.918840,0.000000,0.000000,0.000000
8,mv Shonganiso Clinic,202410,Air temperature (ERA5-Land),0.0,0.510257,0.039037,13.071230,0.923496,0.000000,0.000000,0.000000
9,mv Shonganiso Clinic,202411,Air temperature (ERA5-Land),0.0,0.300497,0.033132,9.069813,0.889744,0.000000,0.000000,0.000000


## 7. Create/extend the two monthly DHIS2 datasets, then publish

The next cell maintains **GHAP Climate-Health Attribution (monthly)** and
**GHAP Climate-Health Forecast (monthly)**. Each health outcome gets a form section;
new outcomes and climate-variable categories are added automatically on the next
refreshed input/model run. Existing sections, members and sharing are preserved.

Attribution includes 13 facility measures, including fitted counts at observed and
reference climate, attributable cases/fractions, relative risks and uncertainty.
Aggregate fractions and uncertainty are stored at their calculated levels.

`CREATE_OR_UPDATE_DATASETS=True` and `METADATA_MODE='live'` create/update metadata.
For values, `PUBLISH_MODE='off'` writes review files; `'dry_run'` validates on DHIS2;
`'live'` uploads reviewed model results (`REVIEWED_BY` must identify their reviewer).
Live mode validates all batches first, reopens affected completed forms, upserts
values, reads every value back, completes and verifies forms, then submits and
tracks aggregate analytics and checks visibility for both datasets.

Analytics is a server-wide aggregate refresh, not restricted to these datasets.
Failures stop the workflow and are recorded in `publish_report.json`. Repeating a
run updates the same value keys; interrupted analytics monitoring resumes its task.
No validation run commits metadata/data, completes forms or triggers analytics.


In [ ]:
from ghap_engine.publish import create_or_update_datasets, publish
import importlib
import ghap_engine.publish_lifecycle as lifecycle
import ghap_engine.publish_workflow as workflow

importlib.reload(lifecycle)
importlib.reload(workflow)

PUBLISH_MODE = globals().get("PUBLISH_MODE", "off")  # Use the selected configuration mode.
REVIEWED_BY = "Obvious J Kawanzaruwa"  # Replace with the person who reviewed the results.
settings["publish_max_values"] = 5_000_000
settings["publish_complete"] = True
settings["publish_analytics"] = True

if CREATE_OR_UPDATE_DATASETS:
    effective_metadata_mode = 'dry_run' if PUBLISH_MODE == 'dry_run' else METADATA_MODE
    metadata_report, output_ids, climate_category_combos = create_or_update_datasets(
        client, data, settings, run.dir, mode=effective_metadata_mode,
    )
    display(pd.Series({
        'status': metadata_report['status'],
        'attribution_dataset_uid': output_ids['datasets']['attribution'],
        'forecast_dataset_uid': output_ids['datasets']['forecast'],
        'health_outcomes': len(data['outcomes']),
        'climate_indicators': len(data['climate_vars']),
    }, name='DHIS2 dataset setup'))

# Live publishing automatically handles update -> verification -> completion -> analytics.
publish_report = publish(
    client, results, forecasts, data, settings, run,
    mode=PUBLISH_MODE, reviewed_by=REVIEWED_BY,
)
display(pd.Series(publish_report, name='DHIS2 publication'))


status                     metadata_ready
attribution_dataset_uid       JCo5G20MVHs
forecast_dataset_uid          u4LeniuSnf2
health_outcomes                        10
climate_indicators                     48
Name: DHIS2 dataset setup, dtype: object

Validated batch 1 (5000 values)
Validated batch 2 (5000 values)
Validated batch 3 (5000 values)
Validated batch 4 (5000 values)
Validated batch 5 (5000 values)
Validated batch 6 (5000 values)
Validated batch 7 (5000 values)
Validated batch 8 (5000 values)
Validated batch 9 (5000 values)
Validated batch 10 (5000 values)
Validated batch 11 (5000 values)
Validated batch 12 (5000 values)
Validated batch 13 (5000 values)
Validated batch 14 (424 values)
Validated batch 15 (5000 values)
Validated batch 16 (5000 values)
Validated batch 17 (5000 values)
Validated batch 18 (5000 values)
Validated batch 19 (5000 values)
Validated batch 20 (5000 values)
Validated batch 21 (5000 values)
Validated batch 22 (5000 values)
Validated batch 23 (5000 values)
Validated batch 24 (5000 values)
Validated batch 25 (5000 values)
Validated batch 26 (5000 values)
Validated batch 27 (5000 values)
Validated batch 28 (5000 values)
Validated batch 29 (5000 values)
Validated batch 30 (5000 values)
Validated batch 31 (

RuntimeError: Some output forms remain complete; data upload was not started.

In [25]:
import importlib
import pandas as pd
import ghap_engine.publish_lifecycle as lifecycle
import ghap_engine.publish_workflow as workflow

importlib.reload(lifecycle)
importlib.reload(workflow)

from ghap_engine.publish import publish

PUBLISH_MODE = "live"
REVIEWED_BY = "Obvious J Kawanzaruwa"

settings["publish_max_values"] = 5_000_000
settings["publish_complete"] = True
settings["publish_analytics"] = False
settings["analytics_skip_outliers"] = False

publish_report = publish(
    client, results, forecasts, data, settings, run,
    mode=PUBLISH_MODE,
    reviewed_by=REVIEWED_BY,
)

display(pd.Series(publish_report, name="DHIS2 publication"))

Validated batch 1/32 (5000 values)
Validated batch 2/32 (5000 values)
Validated batch 3/32 (5000 values)
Validated batch 4/32 (5000 values)
Validated batch 5/32 (5000 values)
Validated batch 6/32 (5000 values)
Validated batch 7/32 (5000 values)
Validated batch 8/32 (5000 values)
Validated batch 9/32 (5000 values)
Validated batch 10/32 (5000 values)
Validated batch 11/32 (5000 values)
Validated batch 12/32 (5000 values)
Validated batch 13/32 (5000 values)
Validated batch 14/32 (424 values)
Validated batch 15/32 (5000 values)
Validated batch 16/32 (5000 values)
Validated batch 17/32 (5000 values)
Validated batch 18/32 (5000 values)
Validated batch 19/32 (5000 values)
Validated batch 20/32 (5000 values)
Validated batch 21/32 (5000 values)
Validated batch 22/32 (5000 values)
Validated batch 23/32 (5000 values)
Validated batch 24/32 (5000 values)
Validated batch 25/32 (5000 values)
Validated batch 26/32 (5000 values)
Validated batch 27/32 (5000 values)
Validated batch 28/32 (5000 values)
Va

mode                                                              live
run_id                                        20261111_084405_5cd2b029
status                                                       published
metadata             {'status': 'metadata_ready', 'datasets': {'att...
data_values                                                     154614
datasets             {'attribution': 'JCo5G20MVHs', 'forecast': 'u4...
validated_batches                                                   32
accepted_batches                                                    32
import               {'imported': 0, 'updated': 154614, 'ignored': ...
readback_verified                                      154614 / 154614
completion                       {'status': 'verified', 'forms': 1337}
Name: DHIS2 publication, dtype: object

## 8. Reload the latest saved tables without rerunning

Run this cell on its own after restarting the kernel.

In [37]:
from pathlib import Path  # Allow this cell to run independently after a kernel restart.
import pandas as pd  # Load the saved tables.
from IPython.display import display  # Display tables in Jupyter.
current = Path('ghap_outputs/current')  # Fixed location of the latest run.
print('Run id:', (current / 'RUN_ID.txt').read_text())  # Identify the run.
saved = {p.stem: p for p in sorted(current.glob('*.parquet'))}  # Every saved table.
print(sorted(saved))  # List table names.
forecast_outcome = pd.read_parquet(saved['forecast_outcome'])  # The 12-month outlook.
model_summary = pd.read_parquet(saved['model_summary'])  # Pair-level results and QA.
display(forecast_outcome.head())  # Inspect the outlook.
# Facility-month attribution is in ghap_outputs/current/facility_month/<outcome_id>.parquet

Run id: 20261111_151556_5cd2b029
['aggregates', 'climate_seasonality', 'climate_trends', 'completeness_facility_month', 'completeness_summary', 'data_dictionary', 'exposure_response', 'forecast_cv_scores', 'forecast_outcome', 'forecast_pair', 'forecast_skill', 'forecast_thresholds', 'funnel', 'hazard_events', 'hazard_thresholds', 'heterogeneity', 'lag_response', 'mapping_validation', 'missingness_bias', 'model_summary', 'negative_control', 'outcome_seasonality', 'outcome_trends', 'reference_sensitivity', 'run_log', 'sensitivity']


,level,org_unit_id,org_unit_name,period,calendar_month,lead_months,forecast_mean,forecast_median,forecast_pi80_low,forecast_pi80_high,forecast_pi95_low,forecast_pi95_high,expected_cases,climate_attributable_cases,climate_attributable_cases_low,climate_attributable_cases_high,climate_attributable_fraction,forecast_denominator,forecast_rate_per_1000,alert_threshold,epidemic_threshold,prob_exceed_alert,prob_exceed_epidemic,risk_level,outcome_id,indicator_name,selected_model,selected_climate_uid,cv_crps_selected,cv_crps_baseline,skill_vs_baseline,selection_p_value,issue_month,run_id,selected_climate_name
0,facility,AD1DBy9TnBQ,mv Shonganiso Clinic,202610,10,1,0.701,0.0,0.0,2.0,0.0,4.000,0.641552,0.560376,0.306010,0.911289,0.873469,4.0,175.25000,5.0,14.880339,0.002,0.000,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,15.913542,18.006122,0.116215,6.977699e-16,202609,20261111_151556_5cd2b029,Heatwave threshold (ERA5-Land)
1,facility,ALAVbX3p5mM,mi Mudzengi Clinic,202610,10,1,0.000,0.0,0.0,0.0,0.0,0.000,0.000000,0.000000,0.000000,0.000000,NaN,11.0,0.00000,0.0,1.000000,0.000,0.000,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,15.913542,18.006122,0.116215,6.977699e-16,202609,20261111_151556_5cd2b029,Heatwave threshold (ERA5-Land)
2,facility,BP0IGornqbn,mw Chinhoyi Provincial Hospital,202610,10,1,0.000,0.0,0.0,0.0,0.0,0.000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,NaN,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,15.913542,18.006122,0.116215,6.977699e-16,202609,20261111_151556_5cd2b029,Heatwave threshold (ERA5-Land)
3,facility,Cv6kmfDxwVs,mi Lalapanzi Clinic,202610,10,1,0.380,0.0,0.0,1.0,0.0,2.000,0.391490,0.361792,0.206949,0.612822,0.924140,8.0,47.50000,0.0,1.965561,0.281,0.075,watch,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,15.913542,18.006122,0.116215,6.977699e-16,202609,20261111_151556_5cd2b029,Heatwave threshold (ERA5-Land)
4,facility,D5gTnvm2lRc,mc Kaitano Clinic,202610,10,1,31.345,23.0,5.0,69.0,1.0,99.025,32.546355,24.113129,12.046913,42.118041,0.740886,114.0,274.95614,14.5,20.211103,0.678,0.546,high,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,15.913542,18.006122,0.116215,6.977699e-16,202609,20261111_151556_5cd2b029,Heatwave threshold (ERA5-Land)


In [ ]:
forecast_outcome

,level,org_unit_id,org_unit_name,period,calendar_month,lead_months,forecast_mean,forecast_median,forecast_pi80_low,forecast_pi80_high,forecast_pi95_low,forecast_pi95_high,expected_cases,climate_attributable_cases,climate_attributable_cases_low,climate_attributable_cases_high,climate_attributable_fraction,forecast_denominator,forecast_rate_per_1000,alert_threshold,epidemic_threshold,prob_exceed_alert,prob_exceed_epidemic,risk_level,outcome_id,indicator_name,selected_model,selected_climate_uid,cv_crps_selected,cv_crps_baseline,skill_vs_baseline,selection_p_value,issue_month,run_id,selected_climate_name
0,facility,AD1DBy9TnBQ,mv Shonganiso Clinic,202610,10,1,0.786,0.0,0.0,2.0,0.0,4.0,0.738983,0.670097,0.424094,0.998744,0.906783,4.0,196.500000,3.25,12.462535,0.029,0.000,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,16.14461,18.193177,0.112601,2.123670e-24,202609,20261110_142903_5cd2b029,Heatwave threshold (ERA5-Land)
1,facility,ALAVbX3p5mM,mi Mudzengi Clinic,202610,10,1,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,11.0,0.000000,0.00,1.000000,0.000,0.000,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,16.14461,18.193177,0.112601,2.123670e-24,202609,20261110_142903_5cd2b029,Heatwave threshold (ERA5-Land)
2,facility,BP0IGornqbn,mw Chinhoyi Provincial Hospital,202610,10,1,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,NaN,low,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,16.14461,18.193177,0.112601,2.123670e-24,202609,20261110_142903_5cd2b029,Heatwave threshold (ERA5-Land)
3,facility,Cv6kmfDxwVs,mi Lalapanzi Clinic,202610,10,1,0.443,0.0,0.0,1.0,0.0,2.0,0.426178,0.405540,0.250902,0.607069,0.951575,8.0,55.375000,0.00,1.965561,0.323,0.089,watch,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,16.14461,18.193177,0.112601,2.123670e-24,202609,20261110_142903_5cd2b029,Heatwave threshold (ERA5-Land)
4,facility,D5gTnvm2lRc,mc Kaitano Clinic,202610,10,1,26.820,21.0,5.0,56.0,1.0,85.0,27.784318,20.995413,12.586584,32.705773,0.755657,114.0,235.263158,16.00,23.959310,0.600,0.447,moderate,malaria_positive_cases_v_4fb09c,Malaria positive cases (VHW & HFs),climate_model,gbKsMzTJfEC,16.14461,18.193177,0.112601,2.123670e-24,202609,20261110_142903_5cd2b029,Heatwave threshold (ERA5-Land)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7771,national,uLFWBf4qXuv,zw Zimbabwe Ministry of Health,202705,5,8,0.000,0.0,0.0,0.0,0.0,0.0,0.002476,0.000000,0.000000,0.000000,0.000000,NaN,NaN,1.00,33.300131,0.000,0.000,low,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,no_climate_baseline,,0.40221,0.402210,0.000000,NaN,202609,20261110_142903_5cd2b029,
7772,national,uLFWBf4qXuv,zw Zimbabwe Ministry of Health,202706,6,9,0.000,0.0,0.0,0.0,0.0,0.0,0.000185,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.00,3.822798,0.000,0.000,low,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,no_climate_baseline,,0.40221,0.402210,0.000000,NaN,202609,20261110_142903_5cd2b029,
7773,national,uLFWBf4qXuv,zw Zimbabwe Ministry of Health,202707,7,10,0.000,0.0,0.0,0.0,0.0,0.0,0.000050,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.00,1.490168,0.000,0.000,low,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,no_climate_baseline,,0.40221,0.402210,0.000000,NaN,202609,20261110_142903_5cd2b029,
7774,national,uLFWBf4qXuv,zw Zimbabwe Ministry of Health,202708,8,11,0.000,0.0,0.0,0.0,0.0,0.0,0.000096,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.00,1.061262,0.000,0.000,low,rdns_cholera_cases_7e5e82,RDNS - Cholera Cases,no_climate_baseline,,0.40221,0.402210,0.000000,NaN,202609,20261110_142903_5cd2b029,


In [38]:
import datetime as dt
forecast_outcome.to_excel(f"Forecast_data_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.xlsx",index=False)
model_summary.to_excel(f"model_summary_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.xlsx",index=False)